# 11회차 실습 노트북 — 딥러닝 모델과 시계열 데이터 분석

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adminmrobo/fintech_data_analysis/blob/main/notebooks/11_deep_learning_timeseries.ipynb)

해설 페이지: [https://adminmrobo.github.io/fintech_data_analysis/11_deep_learning_timeseries.html](https://adminmrobo.github.io/fintech_data_analysis/11_deep_learning_timeseries.html)

위에서부터 셀을 차례로 실행하세요(Shift+Enter). 데이터는 저장소 `data/` 폴더(fintech-data 복사본)에서 읽으며, 무작위 시드를 42로 고정해 해설 페이지의 대결 숫자와 같은 값이 나옵니다.

글 · 안상선 ((주)M-Robo 대표)

In [1]:
# 준비: 라이브러리와 데이터 불러오기 (Colab에서는 그대로 실행하면 됩니다)
import os, urllib.parse, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

BASE = 'https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/data/'
def load(path, **kw):
    """저장소의 data/ 폴더에서 CSV를 읽는다. 내 컴퓨터에 저장소가 있으면 그 파일을, 없으면 GitHub에서 읽는다."""
    for root in ('data', '../data'):
        f = os.path.join(root, path)
        if os.path.exists(f):
            return pd.read_csv(f, **kw)
    return pd.read_csv(BASE + urllib.parse.quote(path), **kw)

RS = 42   # 해설 페이지와 같은 숫자가 나오도록 무작위 시드를 고정

> 이 노트북은 교육용이며 투자 자문이 아닙니다. 주가는 2013~2019년 저장소 데이터입니다.

## 1. 가격과 수익률

In [2]:
p = load('Revolution3/portfolio_data.csv'); p['Date'] = pd.to_datetime(p.Date)
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(p.Date, p.NFLX); ax[0].set_title('NFLX price')
ax[1].plot(p.Date[1:], np.diff(np.log(p.NFLX)) * 100); ax[1].set_title('NFLX daily log return (%)')
plt.show()

## 2. 대결 재현: "내일 = 오늘" vs AR(10)
과거 10일 가격으로 내일 가격을 맞히는 선형 모델(AR)과, 오늘 가격을 그대로 내일 예측으로 쓰는 기준을 마지막 120거래일로 비교합니다.

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
def ar_vs_naive(series, lags=10, ntest=120):
    y = np.asarray(series, float); n = len(y)
    X = np.column_stack([y[lags - i - 1:n - i - 1] for i in range(lags)]); t = y[lags:]
    m = LinearRegression().fit(X[:-ntest], t[:-ntest])
    return mean_absolute_error(t[-ntest:], X[-ntest:, 0]), mean_absolute_error(t[-ntest:], m.predict(X[-ntest:]))
for c in ['NFLX', 'BTC']:
    nv, ar = ar_vs_naive(p[c])
    print(f'{c}: 내일=오늘 MAE {nv:.2f}   AR(10) MAE {ar:.2f}')

NFLX: 내일=오늘 MAE 6.31   AR(10) MAE 6.39
BTC: 내일=오늘 MAE 128.45   AR(10) MAE 137.43


**AI 판단 연습** — 예측 그래프가 실제 선과 "거의 겹친다"면 좋은 모델일까요? 하루씩 늦게 따라가는 선도 겹쳐 보입니다. 기준(내일 = 오늘)과 먼저 비교하세요.